# 02. Árstruktúra és Piaci Szegmentáció

**Cél**: A kőbányai ingatlanpiac strukturális tagozódásának feltárása építési technológia (panel vs. tégla), méretkategóriák, szobaszám és emeleti elhelyezkedés szerint.

**Kutatási kérdés**: Mekkora a tégla építésű lakások felára a panellakásokhoz képest, és hogyan alakul a méretkategória szerinti fajlagos árprémium Kőbánya alpiacain?

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Elemzett eladó lakások száma: {len(elado)} db.")

### 1. Piaci Szegmensek és Technológiai Prémium KPI-k
A panel és tégla technológia alapvető árazási különbségei.

In [ ]:
n_panel = int(elado['is_panel'].sum())
n_tegla = int((elado['is_panel'] == 0).sum())
med_panel = elado[elado['is_panel'] == 1]['nm_ar_huf'].median()
med_tegla = elado[elado['is_panel'] == 0]['nm_ar_huf'].median()
tegla_premium_pct = ((med_tegla - med_panel) / med_panel) * 100

kpi_cards = [
    ("Panel Lakások", f"{n_panel:,} db".replace(',', ' '), f"{n_panel/len(elado)*100:.1f}% kínálat", "#ef4444"),
    ("Tégla Lakások", f"{n_tegla:,} db".replace(',', ' '), f"{n_tegla/len(elado)*100:.1f}% kínálat", "#06b6d4"),
    ("Panel Medián Ár/m²", fmt_huf(med_panel), "Fajlagos ár", "#f87171"),
    ("Tégla Medián Ár/m²", fmt_huf(med_tegla), "Fajlagos ár", "#22d3ee"),
    ("Tégla Prémium", f"+{tegla_premium_pct:.1f}%", "Fajlagos árkülönbözet", "#10b981"),
    ("Prémium Kategória (>80M)", f"{(elado['ar_millio_ft'] > 80).sum()} db", "Felső piaci szegmens", "#8b5cf6")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Panel vs. Tégla Áreloszlások Városrészenként
A hegedűábra (Violin plot) részletesen szemlélteti az építési technológiák közötti árkülönbséget városszerkezeti egységenként.

In [ ]:
elado['Epites_Tipus'] = elado['is_panel'].map({1: 'Panel', 0: 'Tégla / Egyéb'})

fig1 = px.violin(
    elado,
    x='varosresz',
    y='nm_ar_huf',
    color='Epites_Tipus',
    box=True,
    points='all',
    title='Panel vs. Tégla négyzetméterár eloszlások városrészenként',
    labels={'varosresz': 'Városrész', 'nm_ar_huf': 'Ár / m² (HUF)', 'Epites_Tipus': 'Épülettípus'},
    color_discrete_map={'Panel': '#ef4444', 'Tégla / Egyéb': '#06b6d4'},
    template=PLOTLY_TEMPLATE
)
fig1.update_layout(xaxis_tickangle=-30, height=480)
fig1.show()

### 3. Méretkategória és Szobaszám Kereszttábla Hőtérkép
Hogyan alakul a négyzetméterár a lakás alapterülete és szobaszáma szerint?

In [ ]:
# Méretkategóriák képzése
elado['meret_kategoria'] = pd.cut(
    elado['alapterulet_nm'],
    bins=[0, 40, 55, 70, 90, 300],
    labels=['<40 m²', '40-55 m²', '55-70 m²', '70-90 m²', '>90 m²']
)

# Kereszttábla aggregáció
pivot_nmar = elado.pivot_table(
    index='meret_kategoria',
    columns='szobaszam_kategoria',
    values='nm_ar_huf',
    aggfunc='median'
)

fig2 = px.imshow(
    pivot_nmar,
    text_auto='.0f',
    color_continuous_scale='Blues',
    title='Medián Négyzetméterár (Ft/m²) Méret és Szobaszám Mátrixban',
    labels={'x': 'Szobaszám Kategória', 'y': 'Alapterület Kategória', 'color': 'Ár / m²'},
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=420)
fig2.show()

# Emeleti árazási hatás
emelet_stat = elado.groupby(['is_foldszint', 'is_zaroszint'])['nm_ar_huf'].median().reset_index()
emelet_stat['Pozicio'] = emelet_stat.apply(
    lambda r: 'Földszint' if r['is_foldszint']==1 else ('Zárószint' if r['is_zaroszint']==1 else 'Közbenső emelet'), axis=1
)
fig3 = px.bar(
    emelet_stat,
    x='Pozicio',
    y='nm_ar_huf',
    color='Pozicio',
    title='Földszint és Zárószint Árdiszkontja a Közbenső Emeletekhez Képest',
    labels={'Pozicio': 'Emeleti Elhelyezkedés', 'nm_ar_huf': 'Medián Ár / m² (HUF)'},
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=400, showlegend=False)
fig3.show()

### 4. Interaktív Szegmentációs Laboratórium
Dinamikus szűrő a különböző árszegmensek és minimális mintaelemszámok vizsgálatára.

In [ ]:
w_szegmens = widgets.Dropdown(
    options=[('Épülettípus (Panel/Tégla)', 'Epites_Tipus'), ('Állapot Kód', 'allapot_kod'), ('Szobaszám', 'szobaszam_kategoria')],
    value='Epites_Tipus',
    description='Szegmens:'
)
w_min_ar = widgets.IntSlider(min=20, max=120, value=30, description='Min Ár (M):')
w_max_ar = widgets.IntSlider(min=40, max=200, value=150, description='Max Ár (M):')

out_szeg = widgets.Output()

def frissit_szeg(*args):
    sub = elado[(elado['ar_millio_ft'] >= w_min_ar.value) & (elado['ar_millio_ft'] <= w_max_ar.value)]
    with out_szeg:
        clear_output(wait=True)
        col = w_szegmens.value
        agg = sub.groupby(col)['nm_ar_huf'].agg(['count', 'median', 'mean']).reset_index()
        fig = px.bar(
            agg, x=col, y='median', text='count',
            title=f'Medián négyzetméterár {w_szegmens.label} szerint (Szűrt N={len(sub)})',
            template=PLOTLY_TEMPLATE,
            labels={'median': 'Medián Ár/m²', col: w_szegmens.label}
        )
        fig.show()

w_szegmens.observe(frissit_szeg, names='value')
w_min_ar.observe(frissit_szeg, names='value')
w_max_ar.observe(frissit_szeg, names='value')

display(widgets.HBox([w_szegmens, w_min_ar, w_max_ar]))
display(out_szeg)
frissit_szeg()